# Hyperparameter Tuning and Learning Curves

---

## Learning Objectives

By the end of this notebook, you will:

- Distinguish between **parameters** (learned) and **hyperparameters** (set by you)
- Understand the key hyperparameters: learning rate, batch size, hidden size, number of layers, dropout rate
- Read and interpret **learning curves** (train/val loss vs. epochs) to diagnose underfitting and overfitting
- Implement a simple **grid search** over hyperparameter configurations
- Understand the **learning rate finder** concept
- Train multiple configurations and compare results

## Prerequisites

- **DL100**: Neural network fundamentals (loss, optimization, regularization)
- **DL150**: PyTorch foundations (tensors, autograd, training loops)
- **Notebook 01**: MLP for regression
- **Notebook 02**: MLP for classification (MNIST / sklearn digits)

## Table of Contents

1. [Setup and Imports](#1.-Setup-and-Imports)
2. [Parameters vs Hyperparameters](#2.-Parameters-vs-Hyperparameters)
3. [Key Hyperparameters in Deep Learning](#3.-Key-Hyperparameters-in-Deep-Learning)
4. [Data Preparation](#4.-Data-Preparation)
5. [Learning Curves: Diagnosing Under/Overfitting](#5.-Learning-Curves:-Diagnosing-Under/Overfitting)
6. [Grid Search over Hyperparameters](#6.-Grid-Search-over-Hyperparameters)
7. [Learning Rate Finder](#7.-Learning-Rate-Finder)
8. [Comparing Configurations](#8.-Comparing-Configurations)
9. [Common Mistakes and Debugging Tips](#9.-Common-Mistakes-and-Debugging-Tips)
10. [Exercises](#10.-Exercises)

---

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from itertools import product

set_global_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Device: {device}")
print("Setup complete.")

---

## 2. Parameters vs Hyperparameters

A crucial distinction in machine learning:

| Aspect | Parameters | Hyperparameters |
|--------|-----------|----------------|
| **What** | Weights and biases | Settings that control training |
| **Who sets them** | Learned by the optimizer | Set by the practitioner |
| **When** | Updated every training step | Fixed before training begins |
| **Examples** | $\mathbf{W}, \mathbf{b}$ in linear layers | Learning rate $\eta$, batch size, architecture choices |

Formally, given a model $f_{\theta}(\mathbf{x})$ with parameters $\theta$, the optimizer solves:

$$\theta^* = \arg\min_{\theta} \mathcal{L}(\theta; \lambda)$$

where $\lambda$ represents the hyperparameters that control the optimization process and model architecture.

In [ ]:
# Quick example: parameters vs hyperparameters
model_example = nn.Linear(10, 5)

# Parameters: learned values
print("PARAMETERS (learned by optimizer):")
for name, param in model_example.named_parameters():
    print(f"  {name}: shape={param.shape}, requires_grad={param.requires_grad}")

# Hyperparameters: set by us
print("\nHYPERPARAMETERS (set by us):")
hyperparams = {
    "learning_rate": 0.001,
    "batch_size": 64,
    "hidden_dims": (256, 128),
    "dropout_rate": 0.2,
    "num_epochs": 30,
}
for k, v in hyperparams.items():
    print(f"  {k}: {v}")

---

## 3. Key Hyperparameters in Deep Learning

### Learning Rate ($\eta$)

- Controls step size during gradient descent: $\theta_{t+1} = \theta_t - \eta \nabla \mathcal{L}(\theta_t)$
- **Too high**: loss oscillates or diverges
- **Too low**: training is extremely slow, may get stuck in poor local minima
- Typical range: $10^{-5}$ to $10^{-1}$

### Batch Size

- Number of samples per gradient update
- **Small batches** (16-64): noisier gradients, can act as regularization, slower per-epoch
- **Large batches** (256-1024): smoother gradients, faster per-epoch, may generalize worse

### Hidden Size and Number of Layers

- Control model **capacity** (ability to learn complex functions)
- More neurons/layers = more parameters = higher risk of overfitting
- Rule of thumb: start simple, increase complexity only if underfitting

### Dropout Rate

- Probability of zeroing each neuron during training
- Acts as regularization; typical values: 0.1 to 0.5
- Higher dropout = stronger regularization

---

## 4. Data Preparation

We use the same digit classification task from Notebook 02.

In [ ]:
# Load data (MNIST or sklearn digits fallback)
USE_MNIST = False

try:
    from torchvision import datasets
    import os

    data_dir = os.path.join("..", "..", "data")
    mnist_train = datasets.MNIST(data_dir, train=True, download=True)
    mnist_test = datasets.MNIST(data_dir, train=False, download=True)

    X_train_raw = mnist_train.data.numpy().astype(np.float32)
    y_train_raw = mnist_train.targets.numpy()
    X_test_raw = mnist_test.data.numpy().astype(np.float32)
    y_test_raw = mnist_test.targets.numpy()

    img_height, img_width = 28, 28
    USE_MNIST = True
    print(f"Loaded MNIST: {X_train_raw.shape[0]} train, {X_test_raw.shape[0]} test")

except Exception as e:
    print(f"MNIST download failed: {e}")
    print("Falling back to sklearn digits dataset.\n")

    from sklearn.datasets import load_digits

    digits = load_digits()
    X_all = digits.data.astype(np.float32)
    y_all = digits.target

    img_height, img_width = 8, 8

    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X_all.reshape(-1, img_height, img_width), y_all,
        test_size=0.2, random_state=42, stratify=y_all
    )
    print(f"Loaded sklearn digits: {X_train_raw.shape[0]} train, {X_test_raw.shape[0]} test")

n_classes = 10
input_dim = img_height * img_width
print(f"Input dim: {input_dim}, Classes: {n_classes}")

In [ ]:
# Flatten and normalize
X_train_flat = X_train_raw.reshape(X_train_raw.shape[0], -1)
X_test_flat = X_test_raw.reshape(X_test_raw.shape[0], -1)

max_pixel = X_train_flat.max()
X_train_norm = X_train_flat / max_pixel
X_test_norm = X_test_flat / max_pixel

# Train/val split
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_norm, y_train_raw, test_size=0.15, random_state=42, stratify=y_train_raw
)

print(f"Train: {X_train_final.shape[0]}, Val: {X_val.shape[0]}, Test: {X_test_norm.shape[0]}")

In [ ]:
def make_loaders(X_tr, y_tr, X_val, y_val, batch_size=128):
    """Create train and validation DataLoaders."""
    train_ds = TensorDataset(
        torch.tensor(X_tr, dtype=torch.float32),
        torch.tensor(y_tr, dtype=torch.long),
    )
    val_ds = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_val, dtype=torch.long),
    )
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size)
    return train_loader, val_loader

---

## 5. Learning Curves: Diagnosing Under/Overfitting

**Learning curves** plot training and validation metrics (loss, accuracy) vs. epochs.
They are the primary diagnostic tool for understanding model behavior.

### Diagnosis Guide

| Pattern | Diagnosis | Action |
|---------|-----------|--------|
| Both losses high, not decreasing | **Underfitting** | Increase capacity, lower regularization, train longer |
| Train loss low, val loss high/increasing | **Overfitting** | Add regularization, get more data, reduce capacity |
| Both losses low, close together | **Good fit** | Possibly try more epochs to squeeze more |
| Loss oscillating wildly | **Learning rate too high** | Reduce learning rate |

The **generalization gap** is the difference between train and val loss:

$$\Delta_{\text{gap}} = \mathcal{L}_{\text{val}} - \mathcal{L}_{\text{train}}$$

In [ ]:
class MLP(nn.Module):
    """Configurable MLP for classification."""

    def __init__(self, input_dim, n_classes, hidden_dims=(256, 128), dropout=0.0):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, n_classes))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)


def train_model(model, train_loader, val_loader, epochs=30, lr=1e-3, verbose=False):
    """Train model and return history dict."""
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, epochs + 1):
        # Train
        model.train()
        losses, correct, total = [], 0, 0
        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            logits = model(X_b)
            loss = loss_fn(logits, y_b)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
            correct += (logits.argmax(-1) == y_b).sum().item()
            total += y_b.size(0)

        history["train_loss"].append(np.mean(losses))
        history["train_acc"].append(correct / total)

        # Validate
        model.eval()
        losses, correct, total = [], 0, 0
        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(device), y_b.to(device)
                logits = model(X_b)
                losses.append(loss_fn(logits, y_b).item())
                correct += (logits.argmax(-1) == y_b).sum().item()
                total += y_b.size(0)

        history["val_loss"].append(np.mean(losses))
        history["val_acc"].append(correct / total)

        if verbose and (epoch % 10 == 0 or epoch == 1):
            print(f"  Epoch {epoch:3d} | "
                  f"Train Loss: {history['train_loss'][-1]:.4f} Acc: {history['train_acc'][-1]:.4f} | "
                  f"Val Loss: {history['val_loss'][-1]:.4f} Acc: {history['val_acc'][-1]:.4f}")

    return history

In [ ]:
# Demonstrate three regimes: underfitting, good fit, overfitting
train_loader, val_loader = make_loaders(X_train_final, y_train_final, X_val, y_val, batch_size=128)

configs = {
    "Underfitting (tiny model, high dropout)": {"hidden_dims": (8,), "dropout": 0.5, "lr": 1e-4},
    "Good fit (medium model)": {"hidden_dims": (128, 64), "dropout": 0.1, "lr": 1e-3},
    "Overfitting (large model, no regularization)": {"hidden_dims": (512, 256, 128), "dropout": 0.0, "lr": 1e-3},
}

all_histories = {}
EPOCHS = 40

for name, cfg in configs.items():
    print(f"\nTraining: {name}")
    set_global_seed(42)
    model = MLP(input_dim, n_classes,
                hidden_dims=cfg["hidden_dims"],
                dropout=cfg["dropout"]).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")
    h = train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=cfg["lr"], verbose=True)
    all_histories[name] = h

In [ ]:
# Plot learning curves for all three regimes
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ep = range(1, EPOCHS + 1)

for ax, (name, h) in zip(axes, all_histories.items()):
    ax.plot(ep, h["train_loss"], label="Train Loss", linewidth=2)
    ax.plot(ep, h["val_loss"], label="Val Loss", linewidth=2)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(name, fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Annotate the gap
    gap = h["val_loss"][-1] - h["train_loss"][-1]
    ax.annotate(f"Gap: {gap:.3f}", xy=(EPOCHS, h["val_loss"][-1]),
                fontsize=9, color="red", ha="right")

plt.suptitle("Learning Curves: Underfitting vs Good Fit vs Overfitting", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**Interpretation:**

- **Underfitting**: Both losses remain high. The model is too simple to capture the patterns.
- **Good fit**: Both losses decrease and converge close together. The model generalizes well.
- **Overfitting**: Train loss continues to decrease, but val loss starts increasing. The model memorizes training data.

---

## 6. Grid Search over Hyperparameters

**Grid search** exhaustively evaluates all combinations from a predefined set of hyperparameter values.

For $k$ hyperparameters each with $n_i$ values, total configurations = $\prod_{i=1}^{k} n_i$.

**Warning**: Grid search scales exponentially. Keep the search space small.

In [ ]:
# Define search space
search_space = {
    "lr": [1e-2, 1e-3, 1e-4],
    "hidden_dims": [(64,), (128, 64), (256, 128)],
    "dropout": [0.0, 0.2],
}

# Compute total configurations
total_configs = 1
for v in search_space.values():
    total_configs *= len(v)

print(f"Total configurations to evaluate: {total_configs}")
print(f"Search space: {search_space}")

In [ ]:
# Run grid search
SEARCH_EPOCHS = 15  # fewer epochs for speed
results = []

train_loader_search, val_loader_search = make_loaders(
    X_train_final, y_train_final, X_val, y_val, batch_size=128
)

for lr, hidden_dims, dropout in product(
    search_space["lr"], search_space["hidden_dims"], search_space["dropout"]
):
    set_global_seed(42)
    model = MLP(input_dim, n_classes, hidden_dims=hidden_dims, dropout=dropout).to(device)
    h = train_model(model, train_loader_search, val_loader_search,
                    epochs=SEARCH_EPOCHS, lr=lr)

    best_val_acc = max(h["val_acc"])
    final_val_loss = h["val_loss"][-1]
    n_params = sum(p.numel() for p in model.parameters())

    config = {
        "lr": lr,
        "hidden_dims": hidden_dims,
        "dropout": dropout,
        "best_val_acc": best_val_acc,
        "final_val_loss": final_val_loss,
        "n_params": n_params,
    }
    results.append(config)
    print(f"lr={lr:.0e}, hidden={str(hidden_dims):15s}, drop={dropout:.1f} "
          f"| Val Acc: {best_val_acc:.4f} | Val Loss: {final_val_loss:.4f}")

print(f"\nEvaluated {len(results)} configurations.")

In [ ]:
# Find and display the best configuration
results_sorted = sorted(results, key=lambda x: x["best_val_acc"], reverse=True)

print("Top 5 configurations by validation accuracy:\n")
print(f"{'Rank':<5} {'LR':<10} {'Hidden Dims':<20} {'Dropout':<10} {'Val Acc':<10} {'Params':<10}")
print("-" * 65)
for i, cfg in enumerate(results_sorted[:5]):
    print(f"{i+1:<5} {cfg['lr']:<10.0e} {str(cfg['hidden_dims']):<20} "
          f"{cfg['dropout']:<10.1f} {cfg['best_val_acc']:<10.4f} {cfg['n_params']:<10,}")

best = results_sorted[0]
print(f"\nBest config: lr={best['lr']}, hidden_dims={best['hidden_dims']}, dropout={best['dropout']}")
print(f"Best val accuracy: {best['best_val_acc']:.4f}")

---

## 7. Learning Rate Finder

The **learning rate finder** (Smith, 2017) helps identify a good learning rate range:

1. Start with a very small LR (e.g., $10^{-7}$)
2. Train for a few batches, gradually increasing the LR exponentially
3. Record the loss at each step
4. Plot loss vs. LR
5. Pick the LR where loss decreases most steeply (before it starts increasing)

$$\text{lr}_i = \text{lr}_{\min} \cdot \left(\frac{\text{lr}_{\max}}{\text{lr}_{\min}}\right)^{i/N}$$

In [ ]:
def lr_finder(model, train_loader, lr_min=1e-7, lr_max=1.0, num_steps=200):
    """Simple learning rate finder.

    Trains for `num_steps` batches, exponentially increasing the LR.
    Returns lists of (lrs, losses) for plotting.
    """
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr_min)

    # Exponential LR schedule
    gamma = (lr_max / lr_min) ** (1 / num_steps)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

    lrs = []
    losses = []
    best_loss = float("inf")

    data_iter = iter(train_loader)
    model.train()

    for step in range(num_steps):
        # Get next batch (cycle if needed)
        try:
            X_b, y_b = next(data_iter)
        except StopIteration:
            data_iter = iter(train_loader)
            X_b, y_b = next(data_iter)

        X_b, y_b = X_b.to(device), y_b.to(device)

        optimizer.zero_grad()
        logits = model(X_b)
        loss = loss_fn(logits, y_b)
        loss.backward()
        optimizer.step()
        scheduler.step()

        current_lr = optimizer.param_groups[0]["lr"]
        lrs.append(current_lr)
        losses.append(loss.item())

        # Stop early if loss diverges
        if loss.item() < best_loss:
            best_loss = loss.item()
        if loss.item() > best_loss * 10:
            print(f"  Stopping early at step {step}: loss diverged.")
            break

    return lrs, losses


# Run LR finder
set_global_seed(42)
finder_model = MLP(input_dim, n_classes, hidden_dims=(128, 64)).to(device)
train_loader_finder, _ = make_loaders(X_train_final, y_train_final, X_val, y_val, batch_size=128)

lrs, losses = lr_finder(finder_model, train_loader_finder, lr_min=1e-7, lr_max=1.0, num_steps=300)
print(f"Recorded {len(lrs)} steps.")

In [ ]:
# Plot LR finder results
fig, ax = plt.subplots(figsize=(10, 5))

# Smooth the loss for clearer visualization
window = max(1, len(losses) // 20)
smoothed = np.convolve(losses, np.ones(window) / window, mode="valid")
smoothed_lrs = lrs[:len(smoothed)]

ax.plot(smoothed_lrs, smoothed, linewidth=2, color="steelblue")
ax.set_xscale("log")
ax.set_xlabel("Learning Rate (log scale)")
ax.set_ylabel("Loss")
ax.set_title("Learning Rate Finder")
ax.grid(True, alpha=0.3)

# Find the steepest descent point
gradients = np.gradient(smoothed)
steepest_idx = np.argmin(gradients)
suggested_lr = smoothed_lrs[steepest_idx]

ax.axvline(suggested_lr, color="red", linestyle="--", alpha=0.7,
           label=f"Suggested LR: {suggested_lr:.2e}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

print(f"Suggested learning rate: {suggested_lr:.2e}")
print("Tip: Use a LR slightly lower than the steepest point for stable training.")

---

## 8. Comparing Configurations

Let us train the best grid search configuration vs. the LR finder suggestion and compare learning curves.

In [ ]:
# Train two final models and compare
FINAL_EPOCHS = 30
train_loader_final, val_loader_final = make_loaders(
    X_train_final, y_train_final, X_val, y_val, batch_size=128
)

comparison_configs = {
    f"Grid Search Best (lr={best['lr']:.0e})": {
        "hidden_dims": best["hidden_dims"],
        "dropout": best["dropout"],
        "lr": best["lr"],
    },
    f"LR Finder (lr={suggested_lr:.2e})": {
        "hidden_dims": (128, 64),
        "dropout": 0.0,
        "lr": suggested_lr,
    },
}

comparison_histories = {}
for name, cfg in comparison_configs.items():
    print(f"\nTraining: {name}")
    set_global_seed(42)
    model = MLP(input_dim, n_classes,
                hidden_dims=cfg["hidden_dims"],
                dropout=cfg["dropout"]).to(device)
    h = train_model(model, train_loader_final, val_loader_final,
                    epochs=FINAL_EPOCHS, lr=cfg["lr"], verbose=True)
    comparison_histories[name] = h
    print(f"  Best Val Acc: {max(h['val_acc']):.4f}")

In [ ]:
# Plot comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, FINAL_EPOCHS + 1)

for name, h in comparison_histories.items():
    ax1.plot(ep, h["val_loss"], label=name, linewidth=2)
    ax2.plot(ep, h["val_acc"], label=name, linewidth=2)

ax1.set_xlabel("Epoch"); ax1.set_ylabel("Validation Loss")
ax1.set_title("Val Loss Comparison"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Epoch"); ax2.set_ylabel("Validation Accuracy")
ax2.set_title("Val Accuracy Comparison"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 9. Common Mistakes and Debugging Tips

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Not setting random seed before each run | Results not comparable | Call `set_global_seed(42)` before each config |
| Tuning on the test set | Overly optimistic results, data leakage | Always tune on validation set, evaluate on test only once |
| Too many hyperparameters at once | Combinatorial explosion | Start with LR, then architecture, then regularization |
| Training too few epochs during search | Best configs not identified | Use enough epochs to see convergence trends |
| Ignoring learning curves | No insight into model behavior | Always plot train/val curves |
| Confusing LR finder output | Picking the minimum loss point instead of steepest descent | The best LR is where loss *decreases fastest*, not where it is lowest |
| Not using validation set | Cannot detect overfitting | Always split data into train/val/test |

---

## 10. Exercises

### Exercise 1: Find the Best Hyperparameters for MNIST MLP

Design and run your own grid search with the following search space:

- Learning rates: `[5e-3, 1e-3, 5e-4]`
- Hidden dims: `[(128,), (256, 128), (256, 128, 64)]`
- Dropout: `[0.0, 0.1, 0.3]`
- Batch sizes: `[64, 256]`

Train each for 20 epochs. Report the top 3 configurations.

In [ ]:
# ===== EXERCISE 1: Your code here =====
# search_space_ex = {
#     "lr": [5e-3, 1e-3, 5e-4],
#     "hidden_dims": [(128,), (256, 128), (256, 128, 64)],
#     "dropout": [0.0, 0.1, 0.3],
#     "batch_size": [64, 256],
# }
# results_ex = []
# for lr, hidden_dims, dropout, batch_size in product(...):
#     ...
pass

### Exercise 2: Effect of Batch Size on Training Dynamics

Train the same model with batch sizes `[16, 64, 256, 1024]`. Plot all learning curves on the same axes. What do you observe?

In [ ]:
# ===== EXERCISE 2: Your code here =====
# batch_sizes = [16, 64, 256, 1024]
# for bs in batch_sizes:
#     ...
pass

### Exercise 1 -- Solution

In [ ]:
search_space_ex = {
    "lr": [5e-3, 1e-3, 5e-4],
    "hidden_dims": [(128,), (256, 128), (256, 128, 64)],
    "dropout": [0.0, 0.1, 0.3],
    "batch_size": [64, 256],
}

results_ex = []
total = (
    len(search_space_ex["lr"])
    * len(search_space_ex["hidden_dims"])
    * len(search_space_ex["dropout"])
    * len(search_space_ex["batch_size"])
)
print(f"Total configs: {total}")

for i, (lr, hidden_dims, dropout, batch_size) in enumerate(
    product(
        search_space_ex["lr"],
        search_space_ex["hidden_dims"],
        search_space_ex["dropout"],
        search_space_ex["batch_size"],
    )
):
    set_global_seed(42)
    tr_loader, vl_loader = make_loaders(
        X_train_final, y_train_final, X_val, y_val, batch_size=batch_size
    )
    model = MLP(input_dim, n_classes, hidden_dims=hidden_dims, dropout=dropout).to(device)
    h = train_model(model, tr_loader, vl_loader, epochs=20, lr=lr)

    best_val_acc = max(h["val_acc"])
    results_ex.append({
        "lr": lr, "hidden_dims": hidden_dims, "dropout": dropout,
        "batch_size": batch_size, "best_val_acc": best_val_acc,
    })

    if (i + 1) % 10 == 0:
        print(f"  Completed {i+1}/{total}")

# Sort and display top 3
results_ex_sorted = sorted(results_ex, key=lambda x: x["best_val_acc"], reverse=True)
print(f"\nTop 3 configurations:")
for i, cfg in enumerate(results_ex_sorted[:3]):
    print(f"  {i+1}. lr={cfg['lr']:.0e}, hidden={cfg['hidden_dims']}, "
          f"dropout={cfg['dropout']}, bs={cfg['batch_size']} => Val Acc: {cfg['best_val_acc']:.4f}")

### Exercise 2 -- Solution

In [ ]:
batch_sizes = [16, 64, 256, 1024]
bs_histories = {}
BS_EPOCHS = 20

for bs in batch_sizes:
    set_global_seed(42)
    tr_loader, vl_loader = make_loaders(
        X_train_final, y_train_final, X_val, y_val, batch_size=bs
    )
    model = MLP(input_dim, n_classes, hidden_dims=(128, 64)).to(device)
    h = train_model(model, tr_loader, vl_loader, epochs=BS_EPOCHS, lr=1e-3)
    bs_histories[bs] = h
    print(f"Batch size {bs:>5}: Final Val Acc = {h['val_acc'][-1]:.4f}")

# Plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ep = range(1, BS_EPOCHS + 1)

for bs, h in bs_histories.items():
    ax1.plot(ep, h["train_loss"], label=f"BS={bs}", linewidth=2)
    ax2.plot(ep, h["val_acc"], label=f"BS={bs}", linewidth=2)

ax1.set_xlabel("Epoch"); ax1.set_ylabel("Train Loss")
ax1.set_title("Train Loss by Batch Size"); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.set_xlabel("Epoch"); ax2.set_ylabel("Val Accuracy")
ax2.set_title("Val Accuracy by Batch Size"); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservations:")
print("- Smaller batch sizes have noisier training loss but can generalize well.")
print("- Larger batch sizes train faster per epoch but may need LR adjustment.")
print("- The relationship between batch size and generalization is nuanced.")

---

**Next notebook:** [04 -- Data Leakage and Normalization in DL](04_DataLeakage_and_Normalization_in_DL.ipynb)